In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset
df = pd.read_csv("/content/drive/MyDrive/AI_Crime_Prediction/datasets/Weekly_Risk_Dataset.csv")
print("Data loaded successfully. Shape:", df.shape)

Mounted at /content/drive
Data loaded successfully. Shape: (74818, 6)


In [ ]:
# Fixed Risk Label Mapping
risk_mapping = {"Low": 0, "Medium": 1, "High": 2}
df["Risk_Label"] = df["Risk_Label"].map(risk_mapping)

# Ensure data is sorted by time for every grid
df = df.sort_values(["Grid_ID", "Year", "Week"])

In [ ]:
SEQUENCE_LENGTH = 4
sequence_records = []

for grid in df["Grid_ID"].unique():
    grid_data = df[df["Grid_ID"] == grid].sort_values(["Year", "Week"])

    counts = grid_data["Crime_Count"].values
    labels = grid_data["Risk_Label"].values
    years = grid_data["Year"].values
    weeks = grid_data["Week"].values

    if len(counts) <= SEQUENCE_LENGTH:
        continue

    for i in range(len(counts) - SEQUENCE_LENGTH):
        sequence_records.append({
            "Grid_ID": grid,
            "UnitName": grid_data["UnitName"].iloc[0], # Keeps the Police Station Name
            "Year": years[i+SEQUENCE_LENGTH],
            "Week": weeks[i+SEQUENCE_LENGTH],
            "Week1": counts[i],
            "Week2": counts[i+1],
            "Week3": counts[i+2],
            "Week4": counts[i+3],
            "Target": labels[i+SEQUENCE_LENGTH]
        })

sequence_df = pd.DataFrame(sequence_records)
sequence_df.to_csv("/content/drive/MyDrive/AI_Crime_Prediction/datasets/LSTM_Sequences.csv", index=False)
print("Sequences saved! Total rows:", len(sequence_df))

Sequences saved! Total rows: 67875


In [ ]:
# Instead of random split, we use time-based splitting
train_data = sequence_df[sequence_df['Year'] < 2023]
test_data = sequence_df[sequence_df['Year'] >= 2023]

X_train = np.array(train_data[["Week1", "Week2", "Week3", "Week4"]].values, dtype=np.float32)
y_train = train_data["Target"].values

X_test = np.array(test_data[["Week1", "Week2", "Week3", "Week4"]].values, dtype=np.float32)
y_test = test_data["Target"].values

# Save for training
np.save("/content/drive/MyDrive/AI_Crime_Prediction/datasets/X_train.npy", X_train)
np.save("/content/drive/MyDrive/AI_Crime_Prediction/datasets/X_test.npy", X_test)
np.save("/content/drive/MyDrive/AI_Crime_Prediction/datasets/y_train.npy", y_train)
np.save("/content/drive/MyDrive/AI_Crime_Prediction/datasets/y_test.npy", y_test)

print("Training and Testing files saved correctly based on temporal split.")

Training and Testing files saved correctly based on temporal split.
